# Trace Plot & Histogram

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Burn-in, mixing, and empirical density for a double-well target.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/trace-histogram/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: double well p(x) ∝ exp(-(x^4/4 - x^2)), Langevin from x0 = -3.5
steps, burn_in, eta, x0 = 4000, 500, 0.05, -3.5


def score(x):
    return 2 * x - x ** 3


noise = torch.randn(steps, dtype=torch.float64).tolist()
x, trace = x0, []
for z in noise:
    x = x + eta * score(x) + math.sqrt(2 * eta) * z
    trace.append(x)
trace = torch.tensor(trace, dtype=torch.float64)

print("first steps leave the far start quickly:", [round(v, 2) for v in trace[:8].tolist()])
kept = trace[burn_in:]

# ── Compare the histogram of kept samples with the normalized target ────────
grid = torch.linspace(-3, 3, 6001, dtype=torch.float64)
dens = torch.exp(-(0.25 * grid ** 4 - grid ** 2))
dens = dens / torch.trapezoid(dens, grid)
edges = torch.linspace(-2.5, 2.5, 11, dtype=torch.float64)
hist = torch.histc(kept, bins=10, min=-2.5, max=2.5) / len(kept)
print("\n   bin            chain freq   target prob")
for i in range(10):
    sel = (grid >= edges[i]) & (grid < edges[i + 1])
    print(f"[{edges[i].item():+.1f}, {edges[i + 1].item():+.1f})     {hist[i].item():.3f}        {torch.trapezoid(dens[sel], grid[sel]).item():.3f}")

# One correlated chain of 3500 steps gives a lopsided histogram; the autocorrelation shows why.
# ── Autocorrelation: why a long chain is worth fewer independent samples ────
c = kept - kept.mean()
acf = [(c[:-l] * c[l:]).mean().item() / c.var().item() for l in (1, 10, 50, 200)]
print("\nautocorrelation at lags 1, 10, 50, 200:", [round(a, 3) for a in acf])
side = (kept > 0).int()
print("well-to-well switches after burn-in:", int((side[1:] != side[:-1]).sum()))

first steps leave the far start quickly: [-1.61, -1.49, -1.38, -1.46, -1.24, -1.02, -1.64, -2.13]

   bin            chain freq   target prob
[-2.5, -2.0)     0.027        0.023
[-2.0, -1.5)     0.151        0.130
[-1.5, -1.0)     0.199        0.166
[-1.0, -0.5)     0.140        0.108
[-0.5, +0.0)     0.093        0.072
[+0.0, +0.5)     0.080        0.071
[+0.5, +1.0)     0.103        0.108
[+1.0, +1.5)     0.115        0.166
[+1.5, +2.0)     0.077        0.130
[+2.0, +2.5)     0.014        0.023

autocorrelation at lags 1, 10, 50, 200: [0.962, 0.742, 0.352, 0.1]
well-to-well switches after burn-in: 151
